# Comparison of the four error-model configurations — parametric learning curves

Overlays the across-trajectory **mean 90% credible-interval width** of each calibrated MALI parameter
versus analysis year for all four error-model configurations (A, B, C, D), one panel per parameter.

The styling (font size, aspect ratio, colours, titles, `science` style) matches the single-configuration
parametric-learning figure. **To restyle, edit the `EDIT HERE` block below** — e.g. `FONTSIZE`,
`FIGSIZE` (the aspect ratio), `YTOP_FAC`, or the per-configuration colour/linestyle in `SCEN`.

Run from the `notebooks/` directory (paths are relative to it), all cells top to bottom. It reads the
cached `outputs/posteriors_<tag>/total_widths.jld2` (one file per configuration, produced by
`Amery_Future_Learning_Plots.ipynb` — run that notebook once per configuration first) and writes
`outputs/plots/Config_Comparison_Plots/parametric_learning_all_configs.pdf`. The figure is also
shown inline.


In [ ]:
using JLD2, Statistics, PyCall, PyPlot
# matplotlib needs the conda C++ runtime; harmless no-op outside a conda env.
if haskey(ENV, "CONDA_PREFIX")
    using Libdl
    for lib in ("libstdc++.so.6", "libgcc_s.so.1")
        p = joinpath(ENV["CONDA_PREFIX"], "lib", lib); isfile(p) && Libdl.dlopen(p, Libdl.RTLD_GLOBAL)
    end
end

const OUTDIR = "../outputs/plots/Config_Comparison_Plots"; mkpath(OUTDIR)
try; pyimport("scienceplots"); PyPlot.matplotlib.pyplot.style.use(["default","science","no-latex"]); catch; end

# ================================ EDIT HERE =====================================
FONTSIZE = 20.5                                   # master font size (matches the single-configuration figure)
FIGSIZE  = (19, 11.5)                             # (width, height) inches — controls the aspect ratio
YTOP_FAC = 1.08                                   # y-axis headroom above the tallest 2015 baseline width
# one row per configuration: (legend label, posteriors tag, colour, linestyle)
SCEN = [("A (independent, code on)",  "indep_code",   "#1f5fa8", "--"),
        ("B (Wiener, code on)",       "wiener_code",  "#c0392b", "--"),
        ("C (independent, code off)", "indep_nocode", "#1f5fa8", "-"),
        ("D (Wiener, code off)",      "wiener_nocode","#c0392b", "-")]
# ================================================================================

SLOTS = vcat(2015, collect(2030:15:2300))         # 20 analysis-year slots (slot 1 = 2015 baseline)
TITLE_DICT = Dict(1=>L"\sigma_{max}", 2=>L"q", 3=>L"C_{\mu}",
                  4=>L"C_{\phi}", 5=>L"\log(\gamma_0)", 6=>L"\overline{m}")
letters = ["a","b","c","d","e","f"];


In [ ]:
# across-trajectory MEAN of the 90% CI width: total_widths[real, slot, param] -> MW[tag][slot, param]
MW = Dict{String,Matrix{Float64}}()
for (_, tag, _, _) in SCEN
    tw = JLD2.load("../outputs/posteriors_$(tag)/total_widths.jld2", "total_widths")  # 100x20x6
    MW[tag] = [ (col = filter(!isnan, tw[:, i, j]); isempty(col) ? NaN : mean(col)) for i in 1:20, j in 1:6 ]
end
println("loaded configurations: ", join([t for (_, t, _, _) in SCEN], ", "))

In [ ]:
fig, axs = PyPlot.subplots(nrows=2, ncols=3, figsize=FIGSIZE, dpi=300,
    gridspec_kw=Dict("height_ratios"=>[1,1], "width_ratios"=>[1,1,1], "wspace"=>0.19, "hspace"=>0.25))

for j in 1:6
    r = (j-1)÷3 + 1; c = (j-1)%3 + 1; ax = axs[r, c]
    for (lab, tag, col, ls) in SCEN
        ax.plot(SLOTS, MW[tag][:, j], color=col, linestyle=ls, lw=2.4, marker="o", ms=5, label=lab)
    end
    ax.set_title(TITLE_DICT[j], fontsize=1.2*FONTSIZE, pad=10)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2,4), useMathText=true)
    ax.tick_params(axis="both", which="major", labelsize=0.8*FONTSIZE)
    ax.xaxis.offsetText.set_fontsize(0.8*FONTSIZE); ax.yaxis.offsetText.set_fontsize(0.8*FONTSIZE)
    ax.set_ylim(1e-4, YTOP_FAC * maximum(MW[tag][1, j] for (_, tag, _, _) in SCEN))   # tallest 2015 baseline
    ax.set_xlim(2015, 2300); ax.set_xticks([2050,2100,2150,2200,2250,2300])
    ax.annotate("2015", xy=(2015, 0), xycoords=("data","axes fraction"), xytext=(1991,-0.052),
                fontsize=0.8*FONTSIZE)
    ax.grid(true, alpha=0.5, zorder=1)
    ax.text(c == 1 ? -0.13 : -0.10, 1.095, "($(letters[j]))", ha="left", va="top",
            transform=ax.transAxes, fontsize=FONTSIZE)
end

h, l = axs[1,1].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, fontsize=0.8*FONTSIZE, frameon=true, framealpha=1,
           bbox_to_anchor=(0.512, 0.92), bbox_transform=fig.transFigure)
fig.supxlabel("Analysis year", fontsize=1.2*FONTSIZE, y=0.036)
fig.supylabel("Across trajectory mean 90% credible interval width", fontsize=1.2*FONTSIZE, x=0.066)

out = joinpath(OUTDIR, "parametric_learning_all_configs.pdf")
fig.savefig(out); println("Wrote ", out)